# 📒 Bài 1 — Tensor & Autograd

**Tuần 12**

> Cần chạy trước: `pip install torch`

Sau notebook này, `loss.backward()` không còn là phép thuật.

In [ ]:
import torch
import torch.nn as nn
import numpy as np
import matplotlib.pyplot as plt

plt.rcParams["figure.figsize"] = (11, 4)
plt.rcParams["axes.grid"] = True
plt.rcParams["grid.alpha"] = 0.25
plt.rcParams["axes.spines.top"] = False
plt.rcParams["axes.spines.right"] = False

MAU = ["#4269d0", "#efb118", "#3ca951", "#ff725c", "#a463f2"]
MAU_CHINH, MAU_NHAN = MAU[0], MAU[3]

torch.manual_seed(42)
print("PyTorch", torch.__version__)
print("GPU:", "có" if torch.cuda.is_available() else "không (dùng CPU — vẫn đủ cho phase này)")

## 1. Tensor — NumPy array biết tính đạo hàm

In [ ]:
a = torch.tensor([1.0, 2.0, 3.0])
b = torch.zeros(2, 3)
c = torch.randn(2, 3)

for ten, t in [("a", a), ("b", b), ("c", c)]:
    print(f"{ten}: shape={tuple(t.shape)}  ndim={t.ndim}  "
          f"numel={t.numel()}  dtype={t.dtype}")

### ⚠️ Bẫy kiểu dữ liệu — lỗi số 1 của người mới

In [ ]:
print("torch.tensor([1, 2, 3])     ->", torch.tensor([1, 2, 3]).dtype)
print("torch.tensor([1., 2., 3.])  ->", torch.tensor([1., 2., 3.]).dtype)
print()
print("NumPy mặc định:", np.array([1.0, 2.0]).dtype, " <- float64")
print("Torch mặc định:", torch.tensor([1.0, 2.0]).dtype, " <- float32")

In [ ]:
# @loi-co-y  <- ô này CỐ Ý lỗi, dùng để bạn xem thông báo lỗi
# Trộn int64 với float32 -> lỗi. Chạy ô này để thấy thông báo.
torch.tensor([1, 2, 3]) @ torch.randn(3, 2)

`RuntimeError: expected m1 and m2 to have the same dtype` — thấy thông báo này là biết ngay: **một tensor là `int64`, một là `float32`**.

**Cách tránh:** luôn viết dấu chấm (`[1., 2.]`) hoặc chỉ rõ `dtype=torch.float32`.

**Vì sao deep learning dùng `float32` chứ không `float64`?** Nhanh gấp đôi, tốn nửa bộ nhớ, và độ chính xác 32-bit thừa đủ cho mạng neural. Thực tế còn đi xa hơn — model lớn được train bằng `float16`/`bfloat16`.

## 2. Chuyển qua lại với NumPy

In [ ]:
mang = np.array([1.0, 2.0, 3.0], dtype=np.float32)
t = torch.from_numpy(mang)

mang[0] = 999.0
print("Sửa numpy  ->  tensor:", t)      # tensor cũng đổi!
print()
print("→ from_numpy DÙNG CHUNG bộ nhớ. Muốn độc lập thì .clone()")

t2 = torch.from_numpy(mang).clone()
mang[0] = 1.0
print("Sau .clone(), sửa numpy -> tensor:", t2, " (không đổi)")

## 3. ⭐ Autograd — thứ làm nên deep learning

In [ ]:
w = torch.tensor(0.0, requires_grad=True)

loss = (w - 3) ** 2
loss.backward()

print(f"  w        = {float(w.detach())}")
print(f"  loss     = {float(loss.detach())}")
print(f"  w.grad   = {float(w.grad)}      <- PyTorch tự tính")
print(f"  bằng tay = {2 * (0 - 3)}      <- 2(w-3)")

```
   w ──(trừ 3)──▶ u ──(bình phương)──▶ loss
     ◀──────────    ◀──────────────
      du/dw = 1      dloss/du = 2u

   dloss/dw = dloss/du × du/dw = 2u × 1 = 2(w-3)
```

PyTorch **ghi lại** mọi phép tính thành một đồ thị, rồi `.backward()` lần ngược áp dụng **quy tắc chuỗi**.

Đó là toàn bộ nội dung của **backpropagation**. Không có gì huyền bí.

In [ ]:
# Một lần backward() tính đạo hàm theo MỌI biến
x = torch.tensor(2.0, requires_grad=True)
y = torch.tensor(3.0, requires_grad=True)

f = x**2 * y + 3 * y
f.backward()

print(f"  df/dx = {float(x.grad)}   (bằng tay: 2xy = 2·2·3 = 12)")
print(f"  df/dy = {float(y.grad)}   (bằng tay: x²+3 = 4+3 = 7)")

> 🔑 Dù biểu thức có **1 biến** hay **175 tỷ biến** (GPT-3), vẫn chỉ một lệnh `.backward()`. Chi phí backward xấp xỉ 2 lần chi phí forward — bất kể số tham số. Đó là kết quả toán học đẹp và là lý do deep learning khả thi.

## 4. ⚠️⚠️ Bẫy cộng dồn gradient

In [ ]:
w = torch.tensor(0.0, requires_grad=True)

for lan in range(1, 4):
    loss = (w - 3) ** 2
    loss.backward()
    print(f"  Sau backward lần {lan}:  w.grad = {float(w.grad):>6.1f}")

print("\n  Đúng ra phải luôn là -6.0 !")

In [ ]:
w = torch.tensor(0.0, requires_grad=True)

for lan in range(1, 4):
    if w.grad is not None:
        w.grad.zero_()                      # ← XOÁ trước
    loss = (w - 3) ** 2
    loss.backward()
    print(f"  Sau backward lần {lan}:  w.grad = {float(w.grad):>6.1f}")

> 🚨 **Đây là bug phổ biến nhất của người mới học PyTorch.**
>
> `.grad` được **cộng dồn**, không ghi đè. Quên xoá → gradient của các batch trước còn nguyên → model cập nhật theo hướng vô nghĩa. Triệu chứng: loss giảm rất chậm, dao động, hoặc bùng nổ. **Không có thông báo lỗi nào.**
>
> Đó là lý do mọi training loop bắt đầu bằng `optimizer.zero_grad()`.

**Vì sao PyTorch không tự xoá?** Vì có lúc ta *muốn* cộng dồn — kỹ thuật **gradient accumulation** mô phỏng batch lớn trên GPU nhỏ:

```python
for i, (xb, yb) in enumerate(loader):
    loss = ham_loss(model(xb), yb) / so_buoc_gop
    loss.backward()                       # cộng dồn
    if (i + 1) % so_buoc_gop == 0:
        opt.step(); opt.zero_grad()
```

## 5. Gradient descent — bằng tay vs bằng optimizer

In [ ]:
def bang_tay(w0, lr, n):
    w = torch.tensor(w0, requires_grad=True)
    ls = [float(w.detach())]
    for _ in range(n):
        if w.grad is not None:
            w.grad.zero_()
        ((w - 3) ** 2).backward()
        with torch.no_grad():
            w -= lr * w.grad                  # ← tự cập nhật
        ls.append(float(w.detach()))
    return ls


def bang_optimizer(w0, lr, n):
    w = torch.tensor(w0, requires_grad=True)
    opt = torch.optim.SGD([w], lr=lr)
    ls = [float(w.detach())]
    for _ in range(n):
        opt.zero_grad()
        ((w - 3) ** 2).backward()
        opt.step()                            # ← optimizer cập nhật
        ls.append(float(w.detach()))
    return ls


a, b = bang_tay(0.0, 0.1, 5), bang_optimizer(0.0, 0.1, 5)
print("  bằng tay      :", [round(v, 4) for v in a])
print("  bằng optimizer:", [round(v, 4) for v in b])
print("\n  Giống hệt nhau — optimizer chỉ là bản đóng gói của w -= lr*grad")
print("  Và giống hệt kết quả bạn tự tính ở Phase 02!")

In [ ]:
w_range = np.linspace(-2, 8, 200)
fig, axes = plt.subplots(1, 3, figsize=(14, 3.8))

for ax, lr in zip(axes, [0.01, 0.3, 1.05]):
    ls = bang_optimizer(0.0, lr, 25)
    ax.plot(w_range, (w_range - 3) ** 2, alpha=.35, lw=2)
    ax.plot(ls, [(v - 3) ** 2 for v in ls], "o-", color=MAU_NHAN, ms=4, lw=1)
    ax.axvline(3, color="green", ls="--", alpha=.6)
    ax.set_ylim(-2, 60); ax.set_xlim(-3, 9)
    ax.set_title(f"lr = {lr}  →  w cuối = {ls[-1]:.2f}", fontsize=10)

plt.tight_layout(); plt.show()

Y hệt Phase 02: lr quá nhỏ → chậm, vừa → mượt, quá lớn → phân kỳ. Cùng một thuật toán, chỉ khác công cụ.

## 6. ⚠️ Vì sao phải `torch.no_grad()` khi cập nhật?

In [ ]:
w = torch.tensor(0.0, requires_grad=True)
((w - 3) ** 2).backward()

try:
    w -= 0.1 * w.grad          # KHÔNG có no_grad
except RuntimeError as e:
    print("Lỗi:", e)

Bước cập nhật tham số **không phải** phép tính cần đạo hàm — nó là *hành động* dựa trên đạo hàm đã tính xong. `torch.no_grad()` nói với PyTorch điều đó.

## 7. `no_grad` khi đánh giá

In [ ]:
X = torch.randn(4, 3)
w = torch.randn(3, requires_grad=True)

co_grad = X @ w
with torch.no_grad():
    khong_grad = X @ w

print("Bình thường :", co_grad.requires_grad, "| grad_fn:", co_grad.grad_fn is not None)
print("no_grad     :", khong_grad.requires_grad, "| grad_fn:", khong_grad.grad_fn is not None)

**Luôn dùng `torch.no_grad()` khi:** đánh giá trên validation/test · suy luận trong sản xuất.

Lợi ích: nhanh hơn và **tốn ít RAM hơn đáng kể**. Với model lớn, đây là khác biệt giữa *"chạy được"* và *"CUDA out of memory"*.

> ⚠️ `torch.no_grad()` **không** thay thế `model.eval()`:
> - `no_grad` → tắt tính đạo hàm (vấn đề **tốc độ/bộ nhớ**)
> - `eval()` → đổi hành vi Dropout & BatchNorm (vấn đề **đúng/sai**)
>
> Khi đánh giá phải dùng **cả hai**.

## 8. `nn.Module` — cách tổ chức model

In [ ]:
model = nn.Sequential(
    nn.Linear(4, 8),
    nn.ReLU(),
    nn.Linear(8, 2),
)

print(model)
print()
for ten, p in model.named_parameters():
    print(f"  {ten:<12} shape={tuple(p.shape)}  {p.numel():>3} tham số")
print(f"\n  Tổng: {sum(p.numel() for p in model.parameters())} tham số")

`nn.Linear(4, 8)` chính là `input @ W + b` bạn viết ở Phase 02 — với `W` shape `(4,8)` và `b` shape `(8,)`. Số tham số = `4×8 + 8 = 40`.

Mọi tham số tự động có `requires_grad=True`, nên `loss.backward()` tính gradient cho tất cả cùng lúc.

In [ ]:
# Kiểm tra: một forward + backward tính gradient cho MỌI tham số
X = torch.randn(5, 4)
y = torch.randint(0, 2, (5,))

loss = nn.CrossEntropyLoss()(model(X), y)
loss.backward()

for ten, p in model.named_parameters():
    print(f"  {ten:<12} có gradient shape {tuple(p.grad.shape)}")

## 9. Tổng kết

| Khái niệm | Ý nghĩa |
|---|---|
| `requires_grad=True` | Theo dõi tensor này để tính đạo hàm |
| `.backward()` | Lần ngược đồ thị, tính gradient cho mọi biến |
| `.grad` | Nơi lưu gradient — **cộng dồn**, phải xoá |
| `torch.no_grad()` | Tắt autograd — dùng khi đánh giá/suy luận |
| `nn.Module` | Đóng gói tham số + phép tính |

**Ba lỗi phải nhớ:**
```
❌ torch.tensor([1, 2, 3])           → int64, sẽ lỗi kiểu
❌ Quên optimizer.zero_grad()        → gradient cộng dồn, model học sai
❌ Cập nhật tham số ngoài no_grad()  → RuntimeError
```

## ✍️ Bài tập

```bash
pytest tests/phase05/test_ex01.py -v
```